# Phase 3B — Corrected architecture gate 학습

**목적:** corrected protocol로 B1/G1 controls 또는 pair-local H0–H3를 작은 gate부터 실행한다.  
**입력:** Phase 3A corrected manifest와 versioned config.  
**출력:** checkpoint, per-sample gzip predictions, result JSON, runtime manifest, code snapshot.  
기존 output directory는 절대 재사용하지 않는다.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
root = Path(os.environ.get('GRAPH_CLAD_PROJECT_ROOT', Path.cwd())).resolve()
if not (root / 'scripts').is_dir() and Path('/content/Graph-CLaD').is_dir(): root = Path('/content/Graph-CLaD')
os.chdir(root); sys.path.insert(0, str(root)) if str(root) not in sys.path else None
from scripts.research_paths import resolve_research_paths
paths = resolve_research_paths(project_root=root)
corrected_root = paths.artifact_root / 'phase3_holder_action_v1' / 'corrected_protocol_v2'

## Config 선택과 고정 조건
권장 순서는 smoke → three-fold seed-0 → action-alignment → 통과 후보만 seeds 1/2다. 동일 sample/split/loss/checkpoint/threshold/epoch/patience와 near-matched capacity를 유지한다.

In [ ]:
configs = {
    'corrected_smoke': paths.config_root / 'phase3_corrected_smoke_v2.json',
    'corrected_threefold_seed0': paths.config_root / 'phase3_corrected_threefold_seed0_v2.json',
    'pair_local_smoke': paths.config_root / 'phase3_pair_local_temporal_smoke_v1.json',
    'pair_local_threefold_seed0': paths.config_root / 'phase3_pair_local_temporal_threefold_seed0_v1.json',
    'pair_local_action_alignment': paths.config_root / 'phase3_pair_local_temporal_action_alignment_seed0_v1.json',
}
{name: path.exists() for name, path in configs.items()}

In [ ]:
# 진행/완료 상태만 읽는다. 실행 중 result는 status='running'일 수 있다.
known_results = {
    'pair_local_threefold_seed0': corrected_root / 'pair_local_temporal_threefold_seed0_v1' / 'phase3_pair_local_temporal_threefold_seed0_v1.json',
    'pair_local_action_alignment': corrected_root / 'pair_local_temporal_action_alignment_seed0_v1' / 'phase3_pair_local_temporal_action_alignment_seed0_v1.json',
}
status = {}
for name, path in known_results.items():
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        status[name] = {'status': payload.get('status'), 'runs': len(payload.get('results', [])), 'path': str(path)}
    else:
        status[name] = {'status': 'missing', 'path': str(path)}
status

In [ ]:
# 새 실험용. 기존 config/output을 그대로 재실행하지 않는다.
RUN_TRAINING = False
TRAIN_CONFIG = configs['pair_local_action_alignment']
cfg = json.loads(TRAIN_CONFIG.read_text(encoding='utf-8'))
output_root = Path(cfg['artifacts']['output_root'])
train_cmd = [sys.executable, '-u', '-m', 'scripts.phase3.run_corrected_architecture_gate', '--config', str(TRAIN_CONFIG)]
print(' '.join(train_cmd)); print('OUTPUT_ROOT', output_root)
if RUN_TRAINING:
    if output_root.exists():
        raise FileExistsError('Create a new config/protocol/output version before training')
    subprocess.run(train_cmd, check=True)

## 결과 확인과 다음 phase
완료 조건은 result `status=completed`, 예상 run 수, 빈 stderr, checkpoint/prediction/runtime manifest/code snapshot 존재다. 다음은 같은 공식 단계의 `phase_3b_evaluation_and_controls.ipynb`다.